# Project 01 — Heat Transfer in Forensic Science
**ME 3255: Computational Mechanics**

[Original project brief](https://cooperrc.github.io/computational-mechanics/projects/01_Getting-started-project.html)

Use Newton's law of cooling to model a body's temperature and estimate time of death:
\[
\frac{dT}{dt}=-K(T-T_a).
\]
The body is **85°F at 11:00 a.m.**, then **80°F 45 minutes later**. The ambient temperature is constant at **65°F**. Time is measured in hours, with **t = 0 at discovery (11:00 a.m.)**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T0 = 85.0       # temperature at discovery, degrees F
T1 = 80.0       # temperature 45 minutes later, degrees F
Ta = 65.0       # ambient temperature, degrees F
dt_observed = 45 / 60  # hours


## 1. Estimate K with a finite difference

In [3]:
dTdt = (T1 - T0) / dt_observed
K = -dTdt / (T0 - Ta)
print(f"Estimated cooling rate: {dTdt:.6f} °F/hour")
print(f"K = {K:.6f} per hour")


Estimated cooling rate: -6.666667 °F/hour
K = 0.333333 per hour


In [4]:
Kb = -dTdt / (T1 - Ta)
print(f"K based on T1: {Kb:.6f} per hour")

K based on T1: 0.444444 per hour


## 2. Generalize the calculation into a function
The function accepts temperatures at two times, ambient temperature, and elapsed time in hours. It returns the forward-difference estimate of K in inverse hours. Temperatures must use the same scale.


In [6]:
def return_K(T0, T1, Ta, dt):
    """
    Arguments:
    T0 (float): Initial temperature
    T1 (float): Final temperature
    Ta (float): Ambient temperature
    dt (float): Elapsed time

    Returns:
    float: Estimated cooling rate K
    """

    return -(T1 - T0) / (dt * (T0 - Ta))

K = return_K(T0, T1, Ta, dt_observed)
print(f"Function result: K = {K:.6f} per hour")

Function result: K = 0.333333 per hour


## 3a. Show Euler convergence to the analytical solution
For constant K and ambient temperature, the analytical solution is
\[
T(t)=T_a+(T_0-T_a)e^{-Kt}.
\]
Forward Euler advances the temperature using
\[
T_{n+1}=T_n-K\Delta t(T_n-T_a).
\]
We compare both solutions over 12 hours, using the **same estimated K** in each. Halving the time step should approximately halve the error, because forward Euler is first-order accurate.


In [ ]:
t = np.linspace(0, 45/60, 100)  # time in hours
dt = t[1] - t[0]
T = np.zeros_like(t)
for i in range(1, len(t)):
    DTdt = -K * (T[i-1] - Ta)
    T[i] = T[i-1] + DTdt * dt



**Interpretation:** The maximum error decreases as the time step shrinks; the error ratio approaches 2 and the observed order approaches 1. Thus Euler converges to the analytical solution for the chosen K.

## 3b. Long-time temperature
Because K > 0, the exponential term tends to zero as t tends to infinity:
\[
\boxed{\lim_{t\to\infty} T(t)=T_a=65^\circ\mathrm{F}.}
\]

## 3c. Estimate the time of death
Set T(t) = 98.6°F and solve for t:
\[
t=-\frac{1}{K}\ln\left(\frac{98.6-T_a}{T_0-T_a}\right).
\]
This produces a **negative** t because the body was warmer before discovery. Subtract the elapsed interval from 11:00 a.m.


In [5]:
T_death = 98.6
t_death = -np.log((T_death - Ta) / (T0 - Ta)) / K
# The date is arbitrary; only the clock time matters in this problem.
discovery = datetime(2000, 1, 1, 11, 0)
death = discovery + timedelta(hours=float(t_death))
print(f't = {t_death:.6f} hours relative to discovery')
print(f'Time before discovery: {-t_death * 60:.2f} minutes')
print(f'Estimated time of death: {death.strftime("%I:%M:%S %p")} (approximately 9:27 a.m.)')
assert np.isclose(analytical_temperature(t_death), T_death)


t = -1.556381 hours relative to discovery
Time before discovery: 93.38 minutes
Estimated time of death: 09:26:37 AM (approximately 9:27 a.m.)


## Results and model limits
- Forward-difference cooling coefficient: **K = 0.333333 h⁻¹**.
- Euler's method converges with approximately **first-order accuracy**.
- Long-time temperature: **65°F**.
- Estimated time of death: **about 9:27 a.m.**, roughly 93.4 minutes before discovery.

The finite-difference estimate is approximate: its analytical curve need not pass exactly through the second measured temperature. A coefficient fitted directly to the exponential would be a different estimate; the calculations above intentionally use the finite-difference coefficient throughout, as requested. The time-of-death result also assumes a body temperature of 98.6°F at death and that the same idealized cooling model applies before discovery.
